## 5_proxy_removal_examples.ipynb
### Fairness and Compliance of AI
### University of Oklahoma

This script demonstrates how to remove proxies for protected attributes. The latent expression of the protected attribute(s) is removed from the remainder of the features. When we remove the effect of only one attribute, it is done by building a linear regression model to predict each feature from the protected attribute. The new data becomes the original data minus the linear effect of the attribute (residuals).

Removing proxies from a set of potentially related protected attributes is more complicated. First we find the orthogonal of the column matrix that represents the set of protected attributes (__A__). We then project the matrix of the remaining features (__X__) onto the orthogonal.

We use the UCI Adult Dataset for which we previously imputed missing values. We then encode the labels and features appropriately through integer mapping, one-hot encoding, and z-score normalization.

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import os, random

# Set global seeds for reproducibility
os.environ["PYTHONHASHSEED"] = "42"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
random.seed(42)
np.random.seed(42)

# ---------------------------------------------------------
# 1. Load Adult dataset
# ---------------------------------------------------------
df = pd.read_csv("adult_imputed.csv")

# Protected attributes
A1 = df["sex"]
A2 = df["race"]

# Encode protected attributes
A1_enc = LabelEncoder().fit_transform(A1)
A2_enc = LabelEncoder().fit_transform(A2)

# Target variable (classification)
Y_class = df["target"]
Y_le = LabelEncoder()
Y_class_enc = Y_le.fit_transform(Y_class)

# ---------------------------------------------------------
# 2. Build predictor matrix INCLUDING categoricals, EXCLUDING target + protected attrs
# ---------------------------------------------------------
X = df.drop(columns=["sex", "race", "target"])   # <-- critical fix: drop target

numeric_cols = X.select_dtypes(include=[np.number]).columns
categorical_cols = X.select_dtypes(exclude=[np.number]).columns

# Normalize numeric predictors
scaler = StandardScaler()
X_numeric_scaled = pd.DataFrame(
    scaler.fit_transform(X[numeric_cols]),
    columns=numeric_cols,
    index=X.index
)

# One-hot encode categorical predictors
X_categorical = pd.get_dummies(X[categorical_cols], drop_first=True)

# Final encoded predictor matrix
X_enc = pd.concat([X_numeric_scaled, X_categorical], axis=1)


FileNotFoundError: [Errno 2] No such file or directory: 'adult_imputed.csv'

### Function to remove latent effect of a single protected attribute

In [ ]:
# ---------------------------------------------------------
# 3. Helper: remove correlation with a single protected attribute
# ---------------------------------------------------------
def remove_corr_single(X, A):
    """
    Remove linear correlation between each feature in X and protected attribute A.
    Works for numeric + one-hot encoded categorical features.
    """
    X_new = X.copy()
    A = A.reshape(-1, 1)

    for col in X.columns:
        model = LinearRegression().fit(A, X[col])
        pred = model.predict(A)
        X_new[col] = X[col] - pred  # orthogonal projection residual

    return X_new


### Remove the single attribute

In [ ]:
# ---------------------------------------------------------
# 4. Apply single-attribute correlation removal
# ---------------------------------------------------------
X_single = remove_corr_single(X_enc, A1_enc)

print("Original correlation with sex:")
print(X_enc.corrwith(pd.Series(A1_enc)).sort_values().head())

print("\nAfter single-attribute correlation removal:")
print(X_single.corrwith(pd.Series(A1_enc)).sort_values().head())

### Function to remove latent effect of multiple attributes

In [ ]:
# ---------------------------------------------------------
# 5. Helper: remove correlation with multiple protected attributes
# ---------------------------------------------------------
def remove_corr_multi(X, A_mat):
    """
    Remove linear correlation between each feature in X and ALL protected attributes jointly.
    Equivalent to projecting onto the orthogonal complement of span(A_mat).
    """
    X_new = X.copy()

    for col in X.columns:
        model = LinearRegression().fit(A_mat, X[col])
        pred = model.predict(A_mat)
        X_new[col] = X[col] - pred

    return X_new

### Remove multiple attributes

In [ ]:
# ---------------------------------------------------------
# 6. Apply multi-attribute correlation removal
# ---------------------------------------------------------
A_multi = np.column_stack([A1_enc, A2_enc])
X_multi = remove_corr_multi(X_enc, A_multi)

print("\nOriginal mean |corr| with sex and race:")
print(
    X_enc.corrwith(pd.Series(A1_enc)).abs().mean(),
    X_enc.corrwith(pd.Series(A2_enc)).abs().mean()
)

print("\nAfter multi-attribute correlation removal:")
print(
    X_multi.corrwith(pd.Series(A1_enc)).abs().mean(),
    X_multi.corrwith(pd.Series(A2_enc)).abs().mean()
)

### Check to see classification performance with new data
With the effect of _sex_ and _race_ completely removed, we would expect to see a reduction in the performance of income class prediction. We run a classification using the features without the effect of sex to check.

In [ ]:
# ---------------------------------------------------------
# 7. Train/test split
# ---------------------------------------------------------
idx = np.arange(len(df))

X_train, X_test, X_multi_train, X_multi_test,y_train, y_test, idx_train, idx_test = train_test_split(
    X_enc, X_multi, Y_class_enc, idx, test_size=0.3, random_state=42, stratify=Y_class_enc
)

# Use SAME indices to split proxy-removed features
X_multi_train = X_multi.iloc[idx_train]
X_multi_test  = X_multi.iloc[idx_test]

# ---------------------------------------------------------
# 8. Baseline logistic regression (original features)
# ---------------------------------------------------------
clf = LogisticRegression(max_iter=500, solver="lbfgs")
clf.fit(X_train, y_train)

# ---------------------------------------------------------
# 9. Proxy-removed logistic regression
# ---------------------------------------------------------
clf_multi = LogisticRegression(max_iter=500, solver="lbfgs")
clf_multi.fit(X_multi_train, y_train)



In [ ]:
# ---------------------------------------------------------
# 10. Evaluation
# ---------------------------------------------------------
print("\n=== Original Model ===")
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

print("\n=== Proxy-Removed Model ===")
y_pred_multi = clf_multi.predict(X_multi_test)
print(classification_report(y_test, y_pred_multi))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_multi))
